# 기본 RAG 파이프라인 구축

In [1]:
# !pip install langchain langchain-community pypdf
# !pip install -U langchain-text-splitters
# !pip install -U chromadb
#!pip install -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf

In [2]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("company_manual.pdf")
documents = loader.load()

/tmp/ipykernel_9770/2275095145.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

results = retriever.invoke("기내에 가위 반입이 되나요?")

print(results)

[Document(metadata={'page': 2, 'creator': '(unspecified)', 'source': 'company_manual.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'moddate': '2026-09-22T04:07:17+00:00', 'keywords': '', 'total_pages': 4, 'trapped': '/False', 'page_label': '3', 'creationdate': '2026-09-22T04:07:17+00:00'}, page_content='3.3 전자담배 및 리튬배터리 제품\n전자담배 및 니코틴 흡입 기기는 기내 반입은 가능하나 기내에서의 사용(흡연)은 전 노선에서 금지된다.\n위탁 수하물에 넣어 화물칸으로 운송하는 것은 금지된다.\n제4장 기내 반입 금지 물품\n다음 물품은 보안 검색 규정에 따라 기내 반입이 금지되며, 적발 시 탑승구 또는 보안 검색대에서 압수될\n수 있다.\n • 100ml를 초과하는 액체, 젤, 스프레이류 (단, 투명 지퍼백에 담긴 100ml 이하 용기는 예외)\n• 칼날이 있는 도구 일체 (커터칼, 과도, 가위 등, 단 날 길이 6cm 이하 가위는 위탁 수하물로만 허용)\n• 라이터 및 성냥 (1인당 라이터 1개에 한해 소지 반입 가능, 위탁 수하물 반입은 금지)\n• 스포츠용 배트, 골프채 등 둔기로 사용 가능한 물품\n• 폭죽, 인화성 스프레이 등 위험물로 분류되는 물품\n제5장 특별 승객 서비스\n5.1 임산부 승객\n임신 32주 미만은 별도 서류 없이 탑승 가능하며, 32주 이상 36주 미만은 출발 7일 이내 발급된 산부인과\n진단서(탑승 가능 소견 포함)를 지참해야 한다. 36주 이상은 원칙적으로 탑승이 제한된다.\n5.2 유아 및 소아 동반 승객\n만 2세 미만 유아는

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 사내 문서를 기반으로 질문에 답변하는 어시스턴트입니다.
아래 제공된 컨텍스트만을 사용하여 질문에 답변하세요.
컨텍스트에 답변할 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
답변에 사용한 문서의 출처를 명시하세요

컨텍스트:
{context}

질문: {question}

답변:
""")

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("기내에 반입이 안되는 물품은 어떤것이 있나요?")

print(answer)


기내 반입이 금지되는 물품은 다음과 같습니다:
- 100ml를 초과하는 액체, 젤, 스프레이류 (단, 투명 지퍼백에 담긴 100ml 이하 용기는 예외)
- 칼날이 있는 도구 일체 (커터칼, 과도, 가위 등, 단 날 길이 6cm 이하 가위는 위탁 수하물로만 허용)
- 라이터 및 성냥 (1인당 라이터 1개에 한해 소지 반입 가능, 위탁 수하물 반입은 금지)
- 스포츠용 배트, 골프채 등 둔기로 사용 가능한 물품
- 폭죽, 인화성 스프레이 등 위험물로 분류되는 물품

출처: 제4장 기내 반입 금지 물품
